# FastAPI

Modern, high-performance web framework for building REST APIs.

**Install:** `pip install fastapi uvicorn[standard]`  
**Run:** `uvicorn main:app --reload`  
**Docs:** http://127.0.0.1:8000/docs

**In this notebook:**
- Hello World route
- Path and query parameters
- Pydantic request bodies
- Response models
- HTTP exceptions
- Dependency injection
- CRUD pattern
- Routers and async

> **Note:** The cells below define FastAPI apps. To test them, save the code to a `.py` file and run with uvicorn, then use the auto-generated Swagger UI at `/docs`.

## 1. Hello World

In [ ]:
# Save this as main.py and run: uvicorn main:app --reload

from fastapi import FastAPI

app = FastAPI()

@app.get('/')
def read_root():
    return {'message': 'Hello, World!'}

# GET / → {"message": "Hello, World!"}
# FastAPI automatically serialises dicts to JSON

## 2. Path and Query Parameters

In [ ]:
from fastapi import FastAPI

app = FastAPI()

# Path parameter — part of the URL
@app.get('/items/{item_id}')
def read_item(item_id: int):  # int annotation → automatic validation
    return {'item_id': item_id}

# Query parameter — after the ?
@app.get('/search')
def search(q: str, limit: int = 10, active: bool = True):
    return {'q': q, 'limit': limit, 'active': active}

# GET /items/42      → {"item_id": 42}
# GET /items/abc     → 422 Unprocessable Entity
# GET /search?q=api  → {"q": "api", "limit": 10, "active": true}

## 3. Request Body with Pydantic

Pydantic models define the shape and validation of request bodies.

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel, Field

app = FastAPI()

class Item(BaseModel):
    name: str = Field(..., min_length=1, max_length=50)
    price: float = Field(..., gt=0)
    in_stock: bool = True
    tags: list[str] = []

@app.post('/items/', status_code=201)
def create_item(item: Item):
    return {'created': item.model_dump(), 'name_upper': item.name.upper()}

# POST /items/
# Body: {"name": "Widget", "price": 9.99}
# → {"created": {...}, "name_upper": "WIDGET"}
#
# Body: {"name": "", "price": 9.99}
# → 422 Unprocessable Entity (min_length violated)

## 4. Response Models

`response_model` filters and validates the response — extra fields are stripped.

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class UserIn(BaseModel):
    username: str
    password: str   # should never appear in responses!

class UserOut(BaseModel):
    id: int
    username: str
    # password is NOT here

@app.post('/users/', response_model=UserOut, status_code=201)
def create_user(user: UserIn):
    # password is accepted but never returned
    return {'id': 1, 'username': user.username, 'password': user.password}
    #                                             ↑ filtered out by response_model

## 5. HTTP Exceptions

In [ ]:
from fastapi import FastAPI, HTTPException

app = FastAPI()

db = {1: 'Apple', 2: 'Banana', 3: 'Cherry'}

@app.get('/fruits/{fruit_id}')
def get_fruit(fruit_id: int):
    if fruit_id not in db:
        raise HTTPException(
            status_code=404,
            detail=f'Fruit {fruit_id} not found'
        )
    return {'id': fruit_id, 'name': db[fruit_id]}

# GET /fruits/1  → {"id": 1, "name": "Apple"}
# GET /fruits/99 → 404 {"detail": "Fruit 99 not found"}

## 6. Dependency Injection

In [ ]:
from fastapi import FastAPI, Depends, HTTPException, Header

app = FastAPI()

# Pagination dependency
def paginate(skip: int = 0, limit: int = 10):
    return {'skip': skip, 'limit': min(limit, 100)}

@app.get('/items')
def list_items(page=Depends(paginate)):
    return {'items': [], 'pagination': page}

# Auth dependency
def verify_key(x_api_key: str = Header()):
    if x_api_key != 'my-secret':
        raise HTTPException(401, 'Bad API key')
    return x_api_key

@app.get('/private')
def private(key=Depends(verify_key)):
    return {'data': 'secret', 'authenticated_with': key}

## 7. CRUD Pattern

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI()
db: dict[int, dict] = {}
_id = 0

class Note(BaseModel):
    title: str
    body: str

@app.post('/notes/', status_code=201)
def create(note: Note):
    global _id
    _id += 1
    db[_id] = {'id': _id, **note.model_dump()}
    return db[_id]

@app.get('/notes/')
def read_all():
    return list(db.values())

@app.get('/notes/{note_id}')
def read_one(note_id: int):
    if note_id not in db:
        raise HTTPException(404, 'Not found')
    return db[note_id]

@app.put('/notes/{note_id}')
def update(note_id: int, note: Note):
    if note_id not in db:
        raise HTTPException(404, 'Not found')
    db[note_id].update(note.model_dump())
    return db[note_id]

@app.delete('/notes/{note_id}', status_code=204)
def delete(note_id: int):
    if note_id not in db:
        raise HTTPException(404, 'Not found')
    del db[note_id]

## 8. Routers and Async

In [ ]:
from fastapi import FastAPI, APIRouter
import asyncio

app = FastAPI()

# Router — groups related endpoints
router = APIRouter(prefix='/users', tags=['users'])

@router.get('/')
def list_users():
    return [{'id': 1, 'name': 'Alice'}, {'id': 2, 'name': 'Bob'}]

@router.get('/{user_id}')
def get_user(user_id: int):
    return {'id': user_id, 'name': 'Alice'}

app.include_router(router)

# Async endpoint — use await for non-blocking I/O
@app.get('/slow')
async def slow():
    await asyncio.sleep(0.5)   # yields control to event loop
    return {'status': 'done after 0.5s'}

## Practice

| File | Difficulty | Topics |
|---|---|---|
| [01-easy.py](exercises/01-easy.py) | Easy | Routes, path/query params, basic Pydantic |
| [02-medium.py](exercises/02-medium.py) | Medium | CRUD, HTTPException, response models |
| [03-challenge.py](exercises/03-challenge.py) | Challenge | Routers, dependencies, async, auth |

Solutions: [solutions/](solutions/)